# Prototipo RAG sobre guía clínica en PDF
**Asignatura:** Programación Avanzada en Bioinformática - UMA  
**Autores:** Antonio Sánchez Doña, Aissa Omar El Hammouti Chachoui

In [ ]:
%pip install -q langchain langchain-community langchain-huggingface langchain-ollama
%pip install -q chromadb sentence-transformers pypdf langchain-text-splitters

## DIAGNÓSTICO 0 - Verificar qué modelos Ollama están disponibles


In [1]:
import subprocess

resultado = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
if resultado.returncode != 0:
    print("ERROR: Ollama no está instalado o no está corriendo.")
    print(resultado.stderr)
else:
    print("Modelos disponibles en Ollama:")
    print(resultado.stdout)

# phi3.5 sigue instrucciones de "solo contexto" mucho mejor que llama3.2:1b
modelos_preferidos = ['phi3.5:mini', 'phi3.5', 'llama3.2:3b', 'llama3.2:1b', 'llama3.2', 'mistral', 'tinyllama']
MODEL_NAME = None
for m in modelos_preferidos:
    if m in resultado.stdout:
        MODEL_NAME = m
        break

if MODEL_NAME:
    print(f"\n✓ Modelo seleccionado: {MODEL_NAME}")
else:
    print("\n✗ No se encontró ningún modelo compatible.")
    print("  Ejecuta: ollama pull phi3.5:mini")

Modelos disponibles en Ollama:
NAME                ID              SIZE      MODIFIED       
phi3.5:latest       61819fb370a3    2.2 GB    52 minutes ago    
llama3.2:1b         baf6a787fdff    1.3 GB    12 days ago       
tinyllama:latest    2644915ede35    637 MB    12 days ago       


✓ Modelo seleccionado: phi3.5


## DIAGNÓSTICO 1 - Ver texto RAW del PDF antes de limpiar
Esta celda muestra el texto exacto que extrae PyPDFLoader. Así vemos qué artefactos hay que eliminar.

In [2]:
import sys, re, unicodedata, warnings
warnings.filterwarnings('ignore')
sys.path = [p for p in sys.path if 'AppData\\Roaming\\Python' not in p]

from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = 'GPC_429_Diabetes_2_Osteba_resum.pdf'
loader = PyPDFLoader(PDF_PATH)
pages  = loader.load()
print(f'Páginas cargadas: {len(pages)}')

# --- Mostrar texto RAW de la página 10 para ver los artefactos ---
raw = pages[10].page_content
print(f'\n--- TEXTO RAW página 10 (primeros 500 chars) ---')
print(raw[:500])
print(f'\n--- REPR (muestra caracteres Unicode ocultos) ---')
print(repr(raw[:300]))

Páginas cargadas: 141

--- TEXTO RAW página 10 (primeros 500 chars) ---
GUÍA 	 DE 	 PRÁCTICA 	 CLÍNICA 	 SOBRE 	 DIABETES 	 TIPO 	 2	 13
Autoría y colaboraciones
Grupo de Trabajo de la GPC sobre Diabetes tipo 2 
Alicia Cortázar Galarza, endocrinóloga, Hospital de Cruces (Bizkaia)
Pablo Daza Asurmendi, médico de familia, C.S. de Billabona 
(Comarca Gipuzkoa Oeste)
Arritxu Etxeberria Agirre, farmacéutica de Atención Primaria 
(Comarca Gipuzkoa Este)
Patxi Ezkurra Loiola, médico de familia, C.S. Zumaia (Comarca Gipuzkoa Este)
Ignacia Idarreta Mendiola, médica de famili

--- REPR (muestra caracteres Unicode ocultos) ---
'GUÍA \t DE \t PRÁCTICA \t CLÍNICA \t SOBRE \t DIABETES \t TIPO \t 2\t 13\nAutoría y colaboraciones\nGrupo de Trabajo de la GPC sobre Diabetes tipo 2 \nAlicia Cortázar Galarza, endocrinóloga, Hospital de Cruces (Bizkaia)\nPablo Daza Asurmendi, médico de familia, C.S. de Billabona \n(Comarca Gipuzkoa Oeste)\nArritx'


## Paso 1 - Limpieza y chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def limpiar_texto(texto: str) -> str:
    """Limpieza robusta para PDFs clínicos con artefactos de extracción."""
    texto = unicodedata.normalize('NFKC', texto)
    texto = re.sub(r'[^\S\n]+', ' ', texto)
    # Artefacto de índices: palabras separadas por líneas de solo espacio
    texto = re.sub(r'(\w)\n([ ]*\n)+(\w)', r'\1 \3', texto)
    texto = re.sub(r'(?m)^\s*\d{1,3}\s*$', '', texto)
    texto = re.sub(
        r'(?im)^.*(gu[ií]a de pr[aá]ctica cl[ií]nica|gpc|snpns|sistema nacional).*$',
        '', texto
    )
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()

for page in pages:
    page.page_content = limpiar_texto(page.page_content)

# Excluir páginas de bibliografía (últimas ~25 páginas) y páginas cortas.
# La sección de referencias del PDF empieza en torno a la página 115.
# Esos fragmentos solo contienen citas (autor, revista, año) y contaminan el retrieval.
paginas_utiles = [
    p for p in pages
    if len(p.page_content.strip()) > 150
    and p.metadata.get('page', 0) < 115
]

print(f'Páginas totales:         {len(pages)}')
print(f'Páginas con contenido:   {len([p for p in pages if len(p.page_content.strip()) > 150])}')
print(f'Páginas tras excl. biblio: {len(paginas_utiles)}  (se excluyen páginas ≥ 115)')

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 800,
    chunk_overlap = 100,
    separators    = ['\n\n', '\n', '. ', ' '],
)
chunks = splitter.split_documents(paginas_utiles)
print(f'\nFragmentos generados: {len(chunks)}')
print(f'\n--- Ejemplo fragmento 15 ---')
print(chunks[15].page_content)

Páginas totales:         141
Páginas con contenido:   124
Páginas tras excl. biblio: 98  (se excluyen páginas ≥ 115)

Fragmentos generados: 269

--- Ejemplo fragmento 15 ---
Presentación
La práctica asistencial se hace cada vez más compleja por múltiples factores, 
y el incremento exponencial de información científica es uno de los más re-
levantes. 
Para que las decisiones clínicas sean adecuadas, eficientes y seguras, los 
profesionales necesitan actualizar permanentemente sus conocimientos, 
objetivo al que dedican importantes esfuerzos.

(SNS) creó el proyecto GuíaSalud que tiene como objeto final la mejora en 
la toma de decisiones clínicas basadas en la evidencia científica, a través de 
actividades de formación y de la configuración de un registro de Guías de 

explícitos generados por su comité científico, las ha registrado y las ha difun-
dido a través de Internet.
A principios del año 2006 la Dirección General de la Agencia de Calidad


## Paso 2 - Embeddings + ChromaDB 

In [4]:
import shutil, os, gc, time
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

PERSIST_DIR = './chroma_rag_demo'

# En Windows, ChromaDB mantiene los archivos abiertos hasta que se libera la conexión.
# Hay que cerrar el cliente ANTES de poder borrar el directorio.
try:
    vectorstore._client._system.stop()
    del vectorstore
    gc.collect()
    time.sleep(1)
except NameError:
    pass  # Primera ejecución: vectorstore aún no existe

if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)
    print('Índice anterior eliminado.')

# Modelo multilingüe (cubre español, inglés y 50+ idiomas)
print('Cargando modelo de embeddings multilingüe (~120 MB, solo primera vez)...')
embedding_fn = HuggingFaceEmbeddings(
    model_name    = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs  = {'device': 'cpu'},
    encode_kwargs = {'normalize_embeddings': True},
)
print('  OK — modelo cargado')

print(f'Indexando {len(chunks)} fragmentos...')
vectorstore = Chroma.from_documents(
    documents         = chunks,
    embedding         = embedding_fn,
    persist_directory = PERSIST_DIR,
)
n = vectorstore._collection.count()
print(f'  OK — {n} fragmentos en ChromaDB')
if n != len(chunks):
    print(f'  ALERTA: hay duplicados ({n} != {len(chunks)}). Reinicia el kernel y vuelve a ejecutar.')

Índice anterior eliminado.
Cargando modelo de embeddings multilingüe (~120 MB, solo primera vez)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9423.70it/s]


  OK — modelo cargado
Indexando 269 fragmentos...
  OK — 269 fragmentos en ChromaDB


## DIAGNÓSTICO 2 - Verificar recuperación ANTES de conectar el LLM
Si los fragmentos que aparecen aquí no contienen la respuesta, entonces existe un problema de retrieval.  
Si sí la contienen, entonces el problema es el LLM o el prompt.

In [5]:
retriever = vectorstore.as_retriever(
    search_type  = 'mmr',
    search_kwargs= {'k': 5, 'fetch_k': 25},
)

pregunta_test = '¿Cuál es la dosis recomendada de metformina en diabetes tipo 2?'
docs = retriever.invoke(pregunta_test)

print(f'Pregunta: {pregunta_test}')
print(f'Fragmentos recuperados: {len(docs)}\n')
for i, doc in enumerate(docs, 1):
    print(f'--- Fragmento {i} (página {doc.metadata.get("page", "?")}): ---')
    print(doc.page_content)
    print()

# Verificación: ¿algún fragmento menciona metformina?
menciones = sum(1 for d in docs if 'metformina' in d.page_content.lower())
print(f'Fragmentos que mencionan "metformina": {menciones}/{len(docs)}')
if menciones == 0:
    print('  → Problema de retrieval: el modelo no está recuperando fragmentos relevantes.')
    print('  → Comprueba que la limpieza funcionó (celda anterior).')
else:
    print('  → Retrieval OK. Si el LLM no responde bien, el problema es el prompt o el modelo.')

Pregunta: ¿Cuál es la dosis recomendada de metformina en diabetes tipo 2?
Fragmentos recuperados: 5

--- Fragmento 1 (página 40): ---
Figura 2. Algoritmo de tratamiento de la DM 2
(METFORMINA ± SULFONILUREAS) 
+
Intensificar tratamiento con insulina 
en dos o más dosis 
INTERVENCIÓN SOBRE ESTILOS DE VIDA 
(dieta y ejercicio) 
Monoterapia
HbA1c  7% 
METFORMINA a
Doble terapia
METFORMINA b + SULFONILUREA c
INSULINA (NPH) d NOCTURNA 
+
METFORMINA ± SULFONILUREAS e
Tto combinado: 
ADO + INSULINA 
Rechazo a 
insulina
SU + MET
+ GLITAZ. 
3-6 meses 
Puede considerarse una sulfonilurea en 
pacientes sin sobrepeso (IMC < 25)
HbA1c  7% f
HbA1c  7% f, g 
HbA1c  7% f, g 
f
a Si intolerancia a Metformina, utilizar Sulfonilureas.
b Si intolerancia a Metformina, utilizar Glitazonas (preferentemente Pioglitazona).
c Si Sulfonilureas contraindicado o comidas irregulares, utilizar Glinidas (Repaglinida, Nateglinida).

--- Fragmento 2 (página 93): ---
Producto Nombre comercial Dosis adecuada (mg/día)

## Paso 3 - Pipeline RAG completo

In [6]:
from langchain_ollama import OllamaLLM
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate

print(f'Cargando LLM: {MODEL_NAME}...')
llm = OllamaLLM(model=MODEL_NAME, temperature=0, num_predict=800)

test = llm.invoke('Responde solo con la palabra: FUNCIONA')
print(f'  Test LLM → "{test.strip()}"')

PROMPT = """Eres un asistente médico de apoyo para una guía clínica de diabetes tipo 2.

Tu tarea: responder la pregunta usando la información del CONTEXTO siguiente.
- Extrae TODOS los datos relevantes del contexto y formúlalos en una respuesta clara y completa.
- Si el contexto contiene información parcial, úsala e indica qué aspecto no aparece en la guía.
- NO inventes cifras ni datos que no estén en el contexto.

CONTEXTO:
{context}

PREGUNTA: {question}
RESPUESTA:"""

prompt = PromptTemplate(
    template       = PROMPT,
    input_variables= ['context', 'question'],
)

rag_chain = RetrievalQA.from_chain_type(
    llm                     = llm,
    chain_type              = 'stuff',
    retriever               = retriever,
    chain_type_kwargs       = {'prompt': prompt},
    return_source_documents = True,
)
print('Cadena RAG lista.')

Cargando LLM: phi3.5...
  Test LLM → "FUNCIONA"
Cadena RAG lista.


## Demo - Comparativa RAG vs. LLM sin contexto

In [9]:
preguntas = [
    '¿Cuál es la dosis recomendada de metformina en diabetes tipo 2?',
    '¿Cuál es el objetivo de HbA1c en el control de la diabetes tipo 2?',
    '¿Qué efectos adversos tiene la metformina?',
    '¿Qué contraindicaciones tiene la metformina?',
]

for pregunta in preguntas:
    print(f'\n{"="*65}')
    print(f'PREGUNTA: {pregunta}')
    print(f'{"="*65}')

    resultado  = rag_chain.invoke({'query': pregunta})
    fuentes    = sorted({d.metadata.get('page', '?') for d in resultado['source_documents']})
    print(f'\n[CON RAG — páginas fuente: {fuentes}]')
    print(resultado['result'])

    respuesta_sin_rag = llm.invoke(pregunta)
    print(f'\n[SIN RAG — LLM sin contexto]')
    if len(respuesta_sin_rag) > 1500:
        print(respuesta_sin_rag[:1500])
        print('... [respuesta continúa — sin fuente verificable]')
    else:
        print(respuesta_sin_rag)


PREGUNTA: ¿Cuál es la dosis recomendada de metformina en diabetes tipo 2?

[CON RAG — páginas fuente: [35, 40, 52, 93, 114]]
La dosis recomendada de metformina para el tratamiento de la diabetes tipo 2, según el contexto proporcionado, varía entre 850 y 2550 mg al día. Estas dosificaciones están disponibles en medicamentos comerciales como Dianben® 850 mg y Metformina EFG 850 mg. También se menciona una posibilidad de aumentar la dosis hasta un máximo, aunque el contexto no especifica este límite superior exactamente. Es importante notar que los efectos adversos pueden incluir gastritis o intestinales como dolor abdominal, náuseas y diarrea, pero estos síntomas a menudo se atenúan con la alimentación conjunta de la medicación y una titulación gradual de dosis. Además, metformina no provoca hipoglucemia ni aumento de peso. Sin embargo, faltan datos específicos sobre el riesgo de acidosis láctica en pacientes con insuficiencia renal o hepática usando esta medicación.

[SIN RAG — LLM sin

## Evaluación con Precision@5

In [10]:
import pandas as pd

def precision_at_k(pregunta: str, termino: str, k: int = 5) -> float:
    docs = retriever.invoke(pregunta)[:k]
    return sum(1 for d in docs if termino.lower() in d.page_content.lower()) / k

# Mismas preguntas que la demo + términos que SÍ aparecen en el PDF
pares_eval = [
    ('¿Cuál es la dosis recomendada de metformina en diabetes tipo 2?', 'metformina'),
    ('¿Cuál es el objetivo de HbA1c en el control de la diabetes tipo 2?', 'hba1c'),
    ('¿Qué efectos adversos tiene la metformina?',                        'adversos'),
    ('¿Qué contraindicaciones tiene la metformina?',                      'insuficien'),
]

rows = []
for pregunta, termino in pares_eval:
    p = precision_at_k(pregunta, termino)
    rows.append({'Pregunta': pregunta[:52], 'Término proxy': termino, 'P@5': p})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f'\nP@5 media: {df["P@5"].mean():.2f}')
print()
print('P@5 = fracción de los 5 fragmentos recuperados que contienen el término proxy.')
print('Limitación: el PDF usa vocabulario clínico específico; P@5 puede subestimar')
print('el rendimiento real cuando el retrieval es correcto pero usa sinónimos.')

                                            Pregunta Término proxy  P@5
¿Cuál es la dosis recomendada de metformina en diabe    metformina  0.6
¿Cuál es el objetivo de HbA1c en el control de la di         hba1c  0.6
          ¿Qué efectos adversos tiene la metformina?      adversos  0.6
        ¿Qué contraindicaciones tiene la metformina?    insuficien  0.6

P@5 media: 0.60

P@5 = fracción de los 5 fragmentos recuperados que contienen el término proxy.
Limitación: el PDF usa vocabulario clínico específico; P@5 puede subestimar
el rendimiento real cuando el retrieval es correcto pero usa sinónimos.
